# Гипотеза 1: Комбинации и последовательности ФП

**Контрольная точка:** КТ-4 (Промежуточные результаты)  
**Этапы:** Шаг 4 (Комбинации ФП) + Шаг 5 (Последовательности ФП)

---

### Цель ноутбука

Проверить, какие **комбинации** и **последовательности** факторов проблемности (ФП) статистически значимо связаны с дефолтом компании.

### Структура анализа

| № | Раздел | Что выдаёт |
|---|--------|------------|
| 2 | Комбинации ФП | Топ-20 индивидуальных ФП + Топ-10 пар и троек с Lift, Confidence, p-value |
| 3 | Последовательности ФП | Упорядоченные пары/тройки (A→B→C), временные лаги |
| 4 | Анализ по сегментам | Комбинации и последовательности в разрезе X_AREA_RESP |

### Методы

- **Lift** = P(default | FP) / P(default) — во сколько раз ФП повышает вероятность дефолта
- **Confidence** = P(default | FP) — доля дефолтов среди носителей ФП
- **Support** = доля компаний, имеющих данный ФП / комбинацию
- **Fisher exact test** — точная проверка значимости связи ФП→дефолт (для малых выборок)
- **Chi-square test** — проверка значимости (для больших выборок)
- **FDR-коррекция (Benjamini–Hochberg)** — поправка на множественные сравнения

## 0. Импорты и настройки

In [ ]:
import pandas as pd
import numpy as np
import warnings
from itertools import combinations
from scipy.stats import fisher_exact, chi2_contingency
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

## 1. Загрузка данных

Загружаем:
1. **dataset** (wide-формат из EDA) — для анализа комбинаций
2. **CRM** — для извлечения дат ФП (последовательности) и X_AREA_RESP (сегменты)
3. **Дефолты** — для построения референсных дат

In [ ]:
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"

# --- 1. Wide-формат датасет из EDA (компания × ФП) ---
dataset = pd.read_pickle(f"{DATA_DIR}/dataset_fp_default.pkl")
meta_cols = ["inn", "default_flag", "default_date", "severity"]
fp_cols = [c for c in dataset.columns if c not in meta_cols + ["fp_count"]]

print(f"Dataset: {dataset.shape[0]:,} компаний × {len(fp_cols)} ФП")
print(f"Дефолтных: {dataset['default_flag'].sum():,}  "
      f"({dataset['default_flag'].mean():.2%})")

# --- 2. CRM: даты ФП + X_AREA_RESP ---
df_crm = pd.read_csv(
    f"{DATA_DIR}/ECP_CRM_2022_2026.csv",
    sep=";", encoding="utf-8-sig", dtype=str, low_memory=False,
)
df_crm.columns = df_crm.columns.str.strip()
df_crm = df_crm.rename(columns={
    'VAL.1': 'VAL_1', 'DESC_TEXT.1': 'DESC_TEXT_1', 'ROW_ID.1': 'ROW_ID_1',
    'AGREEMENT_NUM.1': 'AGREEMENT_NUM_1', 'ROW_ID.2': 'ROW_ID_2',
    'VAL.2': 'VAL_2', 'NAME.1': 'NAME_1', 'VAL.3': 'VAL_3', 'VAL.5': 'VAL_5',
})
df_crm["IDENTIFICATION_DATE"] = pd.to_datetime(
    df_crm["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce"
)

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]
DATE_FROM = pd.Timestamp("2023-02-01")
DATE_TO   = pd.Timestamp("2025-12-30")
CUTOFF    = pd.Timestamp("2025-12-31")

df_crm = df_crm[
    df_crm["VAL"].str.strip().isin(ALLOWED_SOURCES) &
    (df_crm["IDENTIFICATION_DATE"] >= DATE_FROM) &
    (df_crm["IDENTIFICATION_DATE"] <= DATE_TO)
].copy()

# Длинная таблица ФП с датами
fp_long = df_crm[["X_INN", "NUMBER_FP_SFP", "IDENTIFICATION_DATE"]].copy()
fp_long = fp_long.rename(columns={
    "X_INN": "inn", "NUMBER_FP_SFP": "fp_type", "IDENTIFICATION_DATE": "fp_date",
})
fp_long = fp_long.dropna(subset=["inn", "fp_type", "fp_date"])

# Присоединяем default info + временной фильтр (ФП строго ДО дефолта)
fp_long = fp_long.merge(
    dataset[["inn", "default_flag", "default_date"]], on="inn", how="inner"
)
fp_long["ref_date"] = fp_long["default_date"].fillna(CUTOFF)
fp_long = fp_long[fp_long["fp_date"] < fp_long["ref_date"]].copy()

print(f"CRM (после фильтров): {len(fp_long):,} записей ФП")

# --- 3. X_AREA_RESP — сегмент компании ---
_mode = lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else None
seg_company = (
    df_crm[["X_INN", "X_AREA_RESP"]].dropna(subset=["X_INN"]).drop_duplicates()
    .groupby("X_INN").agg(X_AREA_RESP=("X_AREA_RESP", _mode))
    .reset_index().rename(columns={"X_INN": "inn"})
)
dataset = dataset.merge(seg_company, on="inn", how="left")

print(f"\nX_AREA_RESP:")
print(dataset["X_AREA_RESP"].value_counts(dropna=False).to_string())

# --- 4. Справочник ФП (расшифровка номеров → наименования) ---
ref_book = pd.read_csv(
    "/home/jovyan/documents/DMUKP_FP_DEF/ref_book_fp.csv",
    sep=";", encoding="utf-8-sig", dtype=str,
)
ref_book.columns = ref_book.columns.str.strip()
fp_name_map = dict(zip(ref_book["№"].str.strip(), ref_book["Наименование"].str.strip()))
print(f"Справочник ФП: {len(fp_name_map)} записей")

def fp_label(code):
    """Возвращает 'Код — Наименование' или просто код, если нет в справочнике."""
    name = fp_name_map.get(str(code).strip())
    return f"{code} — {name}" if name else str(code)

base_rate = dataset["default_flag"].mean()
total = len(dataset)
print(f"\nБазовый default rate: {base_rate:.4f}")
print(f"Всего компаний: {total:,}")

## 2. Значимые комбинации ФП

### 2.1 Индивидуальные ФП — статистическая значимость

Для каждого ФП строим таблицу 2×2 (есть ФП / нет ФП) × (дефолт / нет дефолта) и проверяем значимость точным тестом Фишера. Применяем FDR-коррекцию Benjamini–Hochberg для множественных сравнений.

In [ ]:
def compute_fp_stats(data, fp_columns, base_default_rate, min_companies=5):
    """Рассчитывает Lift, Confidence, Support и p-value (Fisher) для каждого ФП."""
    y = data["default_flag"].values
    n = len(data)
    n_def = y.sum()
    n_nodef = n - n_def
    results = []

    for col in fp_columns:
        x = data[col].values
        a = int(((x == 1) & (y == 1)).sum())  # ФП=1, дефолт=1
        b = int(((x == 1) & (y == 0)).sum())  # ФП=1, дефолт=0
        c = int(((x == 0) & (y == 1)).sum())  # ФП=0, дефолт=1
        d = int(((x == 0) & (y == 0)).sum())  # ФП=0, дефолт=0

        n_with = a + b
        if n_with < min_companies:
            continue

        confidence = a / n_with if n_with > 0 else 0
        support = n_with / n
        lift = confidence / base_default_rate if base_default_rate > 0 else 0

        _, p_value = fisher_exact([[a, b], [c, d]], alternative="greater")

        results.append({
            "ФП": col,
            "Компаний с ФП": n_with,
            "Support": round(support, 4),
            "Дефолтов": a,
            "Confidence": round(confidence, 4),
            "Lift": round(lift, 2),
            "p-value (Fisher)": p_value,
        })

    df_res = pd.DataFrame(results)
    if len(df_res) == 0:
        return df_res

    _, fdr_pvals, _, _ = multipletests(df_res["p-value (Fisher)"], method="fdr_bh")
    df_res["p-value (FDR)"] = fdr_pvals
    df_res["Значимый (FDR<0.05)"] = df_res["p-value (FDR)"] < 0.05

    return df_res.sort_values("Lift", ascending=False)


df_individual = compute_fp_stats(dataset, fp_cols, base_rate, min_companies=5)

n_significant = df_individual["Значимый (FDR<0.05)"].sum()
print(f"Всего ФП проверено: {len(df_individual)}")
print(f"Статистически значимых (FDR < 0.05): {n_significant}")

print(f"\n{'='*70}")
print("ТОП-20 ФП по Lift (с ≥5 компаниями):")
print("="*70)
display(
    df_individual.head(20)
    .style.hide(axis="index")
    .format({
        "Confidence": "{:.2%}", "Support": "{:.4f}",
        "p-value (Fisher)": "{:.2e}", "p-value (FDR)": "{:.2e}",
    })
    .bar(subset=["Lift"], color="salmon")
)

### 2.2 Комбинации из 2 ФП

Для каждой пары ФП проверяем: повышается ли вероятность дефолта, когда **оба** ФП присутствуют одновременно. Перебираем пары среди ФП, встречающихся у ≥20 компаний.

In [ ]:
MIN_COMPANIES_COMBO = 10
MIN_DEFAULTS_COMBO = 3

frequent_fp = [c for c in fp_cols if (dataset[c] == 1).sum() >= 20]
print(f"ФП с ≥20 компаниями (для перебора пар): {len(frequent_fp)}")

y = dataset["default_flag"].values
X = dataset[frequent_fp].values
fp_names = np.array(frequent_fp)

pair_results = []
for i, j in combinations(range(len(frequent_fp)), 2):
    mask = (X[:, i] == 1) & (X[:, j] == 1)
    n_with = mask.sum()
    if n_with < MIN_COMPANIES_COMBO:
        continue
    n_def_with = y[mask].sum()
    if n_def_with < MIN_DEFAULTS_COMBO:
        continue

    n_without = (~mask).sum()
    n_def_without = y[~mask].sum()

    a, b = int(n_def_with), int(n_with - n_def_with)
    c, d = int(n_def_without), int(n_without - n_def_without)

    confidence = a / n_with
    support = n_with / len(dataset)
    lift = confidence / base_rate

    _, p_val = fisher_exact([[a, b], [c, d]], alternative="greater")

    pair_results.append({
        "ФП_1": fp_names[i],
        "ФП_2": fp_names[j],
        "Компаний": n_with,
        "Support": round(support, 4),
        "Дефолтов": a,
        "Confidence": round(confidence, 4),
        "Lift": round(lift, 2),
        "p-value (Fisher)": p_val,
    })

df_pairs = pd.DataFrame(pair_results)
if len(df_pairs) > 0:
    _, fdr_pvals, _, _ = multipletests(df_pairs["p-value (Fisher)"], method="fdr_bh")
    df_pairs["p-value (FDR)"] = fdr_pvals
    df_pairs["Значимый (FDR<0.05)"] = df_pairs["p-value (FDR)"] < 0.05
    df_pairs = df_pairs.sort_values("Lift", ascending=False)

n_sig_pairs = df_pairs["Значимый (FDR<0.05)"].sum() if len(df_pairs) > 0 else 0
print(f"Найдено пар: {len(df_pairs)}, значимых (FDR<0.05): {n_sig_pairs}")

print(f"\n{'='*70}")
print("ТОП-10 пар ФП по Lift:")
print("="*70)
if len(df_pairs) > 0:
    display(
        df_pairs.head(10)
        .style.hide(axis="index")
        .format({
            "Confidence": "{:.2%}", "Support": "{:.4f}",
            "p-value (Fisher)": "{:.2e}", "p-value (FDR)": "{:.2e}",
        })
        .bar(subset=["Lift"], color="salmon")
    )
else:
    print("  Пар, удовлетворяющих порогам, не найдено.")

### 2.3 Комбинации из 3 ФП

Перебор троек среди ТОП-30 ФП по индивидуальному Lift (C(30,3) = 4060 комбинаций).

In [ ]:
top_individual = (
    df_individual[df_individual["Компаний с ФП"] >= 20]
    .sort_values("Lift", ascending=False)
    .head(30)["ФП"].tolist()
)
top_idx = [i for i, name in enumerate(frequent_fp) if name in top_individual]
print(f"ТОП-{len(top_idx)} ФП для перебора троек → C({len(top_idx)},3) = "
      f"{len(top_idx)*(len(top_idx)-1)*(len(top_idx)-2)//6} комбинаций")

triple_results = []
for i, j, k in combinations(top_idx, 3):
    mask = (X[:, i] == 1) & (X[:, j] == 1) & (X[:, k] == 1)
    n_with = mask.sum()
    if n_with < MIN_COMPANIES_COMBO:
        continue
    n_def_with = y[mask].sum()
    if n_def_with < MIN_DEFAULTS_COMBO:
        continue

    n_without = (~mask).sum()
    n_def_without = y[~mask].sum()

    a, b = int(n_def_with), int(n_with - n_def_with)
    c, d = int(n_def_without), int(n_without - n_def_without)

    confidence = a / n_with
    support = n_with / len(dataset)
    lift = confidence / base_rate

    _, p_val = fisher_exact([[a, b], [c, d]], alternative="greater")

    triple_results.append({
        "ФП_1": fp_names[i],
        "ФП_2": fp_names[j],
        "ФП_3": fp_names[k],
        "Компаний": n_with,
        "Support": round(support, 4),
        "Дефолтов": a,
        "Confidence": round(confidence, 4),
        "Lift": round(lift, 2),
        "p-value (Fisher)": p_val,
    })

df_triples = pd.DataFrame(triple_results)
if len(df_triples) > 0:
    _, fdr_pvals, _, _ = multipletests(df_triples["p-value (Fisher)"], method="fdr_bh")
    df_triples["p-value (FDR)"] = fdr_pvals
    df_triples["Значимый (FDR<0.05)"] = df_triples["p-value (FDR)"] < 0.05
    df_triples = df_triples.sort_values("Lift", ascending=False)

n_sig_triples = df_triples["Значимый (FDR<0.05)"].sum() if len(df_triples) > 0 else 0
print(f"Найдено троек: {len(df_triples)}, значимых (FDR<0.05): {n_sig_triples}")

print(f"\n{'='*70}")
print("ТОП-10 троек ФП по Lift:")
print("="*70)
if len(df_triples) > 0:
    display(
        df_triples.head(10)
        .style.hide(axis="index")
        .format({
            "Confidence": "{:.2%}", "Support": "{:.4f}",
            "p-value (Fisher)": "{:.2e}", "p-value (FDR)": "{:.2e}",
        })
        .bar(subset=["Lift"], color="salmon")
    )
else:
    print("  Троек, удовлетворяющих порогам, не найдено.")

### 2.4 Сводная таблица комбинаций с наименованиями ФП

Расшифровка номеров ФП через справочник `ref_book_fp`. Объединяем индивидуальные ФП, пары и тройки в единые таблицы с человекочитаемыми названиями.

In [ ]:
# === Индивидуальные ФП с наименованиями ===
df_ind_named = df_individual.copy()
df_ind_named.insert(1, "Наименование", df_ind_named["ФП"].apply(
    lambda x: fp_name_map.get(str(x).strip(), "")
))

print("="*90)
print("ТОП-20 индивидуальных ФП (с наименованиями):")
print("="*90)
display(
    df_ind_named.head(20)
    [["ФП", "Наименование", "Компаний с ФП", "Дефолтов", "Confidence", "Lift",
      "p-value (FDR)", "Значимый (FDR<0.05)"]]
    .style.hide(axis="index")
    .format({"Confidence": "{:.2%}", "p-value (FDR)": "{:.2e}"})
    .bar(subset=["Lift"], color="salmon")
)

# === Пары ФП с наименованиями ===
if len(df_pairs) > 0:
    df_pairs_named = df_pairs.copy()
    df_pairs_named["Наименование_1"] = df_pairs_named["ФП_1"].apply(
        lambda x: fp_name_map.get(str(x).strip(), "")
    )
    df_pairs_named["Наименование_2"] = df_pairs_named["ФП_2"].apply(
        lambda x: fp_name_map.get(str(x).strip(), "")
    )
    df_pairs_named["Комбинация (названия)"] = (
        df_pairs_named["ФП_1"] + " — " + df_pairs_named["Наименование_1"]
        + "\n + " +
        df_pairs_named["ФП_2"] + " — " + df_pairs_named["Наименование_2"]
    )

    print(f"\n{'='*90}")
    print("ТОП-10 пар ФП (с наименованиями):")
    print("="*90)
    display(
        df_pairs_named.head(10)
        [["ФП_1", "Наименование_1", "ФП_2", "Наименование_2",
          "Компаний", "Дефолтов", "Confidence", "Lift",
          "p-value (FDR)", "Значимый (FDR<0.05)"]]
        .style.hide(axis="index")
        .format({"Confidence": "{:.2%}", "p-value (FDR)": "{:.2e}"})
        .bar(subset=["Lift"], color="salmon")
    )

# === Тройки ФП с наименованиями ===
if len(df_triples) > 0:
    df_tri_named = df_triples.copy()
    for col in ["ФП_1", "ФП_2", "ФП_3"]:
        df_tri_named[col.replace("ФП", "Наим")] = df_tri_named[col].apply(
            lambda x: fp_name_map.get(str(x).strip(), "")
        )

    print(f"\n{'='*90}")
    print("ТОП-10 троек ФП (с наименованиями):")
    print("="*90)
    display(
        df_tri_named.head(10)
        [["ФП_1", "Наим_1", "ФП_2", "Наим_2", "ФП_3", "Наим_3",
          "Компаний", "Дефолтов", "Confidence", "Lift",
          "p-value (FDR)", "Значимый (FDR<0.05)"]]
        .style.hide(axis="index")
        .format({"Confidence": "{:.2%}", "p-value (FDR)": "{:.2e}"})
        .bar(subset=["Lift"], color="salmon")
    )

## 3. Последовательности ФП (с учётом порядка возникновения)

Если в п.2 мы проверяли **наличие** комбинации ФП (A и B), то здесь учитываем **порядок**: A→B ≠ B→A.

Для каждой компании:
1. Сортируем все ФП по `IDENTIFICATION_DATE`
2. Извлекаем упорядоченные пары (A→B) и тройки (A→B→C)
3. Сравниваем частоту и дефолтность прямых/обратных последовательностей
4. Считаем временные лаги между ФП

### 3.1 Построение цепочек ФП

In [ ]:
# Для каждой компании: упорядоченный список (fp_type, fp_date), дедупликация по типу ФП
# Если один тип ФП встречался несколько раз — берём самую раннюю дату
fp_first = (
    fp_long
    .sort_values("fp_date")
    .groupby(["inn", "fp_type"], as_index=False)
    .agg(fp_date=("fp_date", "min"))
)

# Кол-во уникальных ФП на компанию
fp_per_inn = fp_first.groupby("inn")["fp_type"].count()
print(f"Компаний с ≥2 различными ФП: {(fp_per_inn >= 2).sum():,} "
      f"(из {len(fp_per_inn):,})")
print(f"Компаний с ≥3 различными ФП: {(fp_per_inn >= 3).sum():,}")

# Присоединяем default_flag
fp_first = fp_first.merge(
    dataset[["inn", "default_flag"]], on="inn", how="inner"
)

### 3.2 Упорядоченные пары (A → B)

Для каждой пары типов ФП считаем: у скольких компаний A возник **раньше** B. Затем проверяем связь с дефолтом.

In [ ]:
# Self-join: все упорядоченные пары (A→B) внутри каждой компании
ordered_pairs = fp_first.merge(fp_first, on=["inn", "default_flag"], suffixes=("_1", "_2"))
ordered_pairs = ordered_pairs[
    (ordered_pairs["fp_type_1"] != ordered_pairs["fp_type_2"]) &
    (ordered_pairs["fp_date_1"] < ordered_pairs["fp_date_2"])
].copy()
ordered_pairs["lag_days"] = (ordered_pairs["fp_date_2"] - ordered_pairs["fp_date_1"]).dt.days

print(f"Всего упорядоченных пар (A→B): {len(ordered_pairs):,}")

# Агрегируем: для каждой упорядоченной пары — сколько компаний, сколько дефолтов
pair_seq_stats = (
    ordered_pairs
    .groupby(["fp_type_1", "fp_type_2"])
    .agg(
        n_companies=("inn", "nunique"),
        n_defaults=("default_flag", "sum"),
        median_lag=("lag_days", "median"),
        mean_lag=("lag_days", "mean"),
    )
    .reset_index()
)
pair_seq_stats = pair_seq_stats[pair_seq_stats["n_companies"] >= MIN_COMPANIES_COMBO].copy()
pair_seq_stats["confidence"] = pair_seq_stats["n_defaults"] / pair_seq_stats["n_companies"]
pair_seq_stats["lift"] = pair_seq_stats["confidence"] / base_rate
pair_seq_stats["support"] = pair_seq_stats["n_companies"] / total

# Fisher test для каждой упорядоченной пары
p_vals = []
for _, row in pair_seq_stats.iterrows():
    a = int(row["n_defaults"])
    b = int(row["n_companies"] - a)
    c = int(dataset["default_flag"].sum() - a)
    d = int(total - a - b - c)
    if d < 0:
        d = 0
    _, p = fisher_exact([[a, b], [c, d]], alternative="greater")
    p_vals.append(p)
pair_seq_stats["p-value (Fisher)"] = p_vals

if len(pair_seq_stats) > 0:
    _, fdr, _, _ = multipletests(pair_seq_stats["p-value (Fisher)"], method="fdr_bh")
    pair_seq_stats["p-value (FDR)"] = fdr
    pair_seq_stats["Значимый"] = fdr < 0.05
    pair_seq_stats = pair_seq_stats.sort_values("lift", ascending=False)

n_sig = pair_seq_stats["Значимый"].sum() if len(pair_seq_stats) > 0 else 0
print(f"Упорядоченных пар (≥{MIN_COMPANIES_COMBO} компаний): {len(pair_seq_stats)}, "
      f"значимых: {n_sig}")

print(f"\n{'='*70}")
print("ТОП-15 упорядоченных пар (A → B) по Lift:")
print("="*70)
if len(pair_seq_stats) > 0:
    show = pair_seq_stats.head(15).copy()
    show["Последовательность"] = show["fp_type_1"].apply(fp_label) + " → " + show["fp_type_2"].apply(fp_label)
    display(
        show[["Последовательность", "n_companies", "support", "n_defaults",
              "confidence", "lift", "median_lag", "mean_lag",
              "p-value (Fisher)", "p-value (FDR)", "Значимый"]]
        .style.hide(axis="index")
        .format({
            "confidence": "{:.2%}", "support": "{:.4f}",
            "median_lag": "{:.0f}", "mean_lag": "{:.0f}",
            "p-value (Fisher)": "{:.2e}", "p-value (FDR)": "{:.2e}",
        })
        .bar(subset=["lift"], color="salmon")
    )

### 3.3 Сравнение прямой и обратной последовательности

Для значимых пар проверяем: имеет ли значение порядок? Сравниваем lift(A→B) vs lift(B→A).

In [ ]:
# Объединяем прямые и обратные пары для сравнения
if len(pair_seq_stats) > 0:
    fwd = pair_seq_stats[["fp_type_1", "fp_type_2", "n_companies", "confidence", "lift", "median_lag"]].copy()
    fwd.columns = ["A", "B", "n_AB", "conf_AB", "lift_AB", "lag_AB"]

    rev = pair_seq_stats[["fp_type_1", "fp_type_2", "n_companies", "confidence", "lift", "median_lag"]].copy()
    rev.columns = ["B", "A", "n_BA", "conf_BA", "lift_BA", "lag_BA"]

    comparison = fwd.merge(rev, on=["A", "B"], how="inner")
    comparison["lift_diff"] = comparison["lift_AB"] - comparison["lift_BA"]
    comparison["порядок_важен"] = comparison["lift_diff"].abs() > 0.5
    comparison = comparison.sort_values("lift_diff", key=abs, ascending=False)

    n_order_matters = comparison["порядок_важен"].sum()
    print(f"Пар с обеими направлениями: {len(comparison)}")
    print(f"Пар, где порядок существенно влияет (|Δlift| > 0.5): {n_order_matters}")

    if len(comparison) > 0:
        print(f"\n{'='*70}")
        print("ТОП-10 пар с наибольшей разницей lift(A→B) vs lift(B→A):")
        print("="*70)
        show = comparison.head(10).copy()
        show["Пара"] = show["A"].apply(fp_label) + " ↔ " + show["B"].apply(fp_label)
        display(
            show[["Пара", "n_AB", "lift_AB", "lag_AB", "n_BA", "lift_BA", "lag_BA", "lift_diff"]]
            .rename(columns={
                "n_AB": "N(A→B)", "lift_AB": "Lift(A→B)", "lag_AB": "Лаг(A→B)",
                "n_BA": "N(B→A)", "lift_BA": "Lift(B→A)", "lag_BA": "Лаг(B→A)",
                "lift_diff": "ΔLift",
            })
            .style.hide(axis="index")
            .format({"Лаг(A→B)": "{:.0f}", "Лаг(B→A)": "{:.0f}"})
        )

### 3.4 Упорядоченные тройки (A → B → C)

Перебираем тройки среди ТОП-20 ФП по индивидуальному Lift.

In [ ]:
# Строим упорядоченные тройки через тройной self-join
# Ограничиваем: только ТОП-20 ФП по lift (иначе слишком долго)
top20_fp = (
    df_individual[df_individual["Компаний с ФП"] >= 20]
    .head(20)["ФП"].tolist()
)
fp3 = fp_first[fp_first["fp_type"].isin(top20_fp)].copy()

triple_seq = (
    fp3.merge(fp3, on=["inn", "default_flag"], suffixes=("_1", "_2"))
    .query("fp_type_1 != fp_type_2 and fp_date_1 < fp_date_2")
    .merge(fp3.rename(columns={"fp_type": "fp_type_3", "fp_date": "fp_date_3"}),
           on=["inn", "default_flag"])
)
triple_seq = triple_seq[
    (triple_seq["fp_type_3"] != triple_seq["fp_type_1"]) &
    (triple_seq["fp_type_3"] != triple_seq["fp_type_2"]) &
    (triple_seq["fp_date_2"] < triple_seq["fp_date_3"])
].copy()

triple_seq["total_lag"] = (triple_seq["fp_date_3"] - triple_seq["fp_date_1"]).dt.days

triple_seq_stats = (
    triple_seq
    .groupby(["fp_type_1", "fp_type_2", "fp_type_3"])
    .agg(
        n_companies=("inn", "nunique"),
        n_defaults=("default_flag", "sum"),
        median_lag=("total_lag", "median"),
    )
    .reset_index()
)
triple_seq_stats = triple_seq_stats[triple_seq_stats["n_companies"] >= MIN_COMPANIES_COMBO].copy()

if len(triple_seq_stats) > 0:
    triple_seq_stats["confidence"] = triple_seq_stats["n_defaults"] / triple_seq_stats["n_companies"]
    triple_seq_stats["lift"] = triple_seq_stats["confidence"] / base_rate
    triple_seq_stats = triple_seq_stats.sort_values("lift", ascending=False)

    print(f"Упорядоченных троек (≥{MIN_COMPANIES_COMBO} компаний): {len(triple_seq_stats)}")

    print(f"\n{'='*70}")
    print("ТОП-10 упорядоченных троек (A → B → C) по Lift:")
    print("="*70)
    show = triple_seq_stats.head(10).copy()
    show["Цепочка"] = show["fp_type_1"].apply(fp_label) + " → " + show["fp_type_2"].apply(fp_label) + " → " + show["fp_type_3"].apply(fp_label)
    display(
        show[["Цепочка", "n_companies", "n_defaults", "confidence", "lift", "median_lag"]]
        .rename(columns={
            "n_companies": "Компаний", "n_defaults": "Дефолтов",
            "confidence": "Confidence", "lift": "Lift", "median_lag": "Медиана лага (дни)",
        })
        .style.hide(axis="index")
        .format({"Confidence": "{:.2%}", "Медиана лага (дни)": "{:.0f}"})
        .bar(subset=["Lift"], color="salmon")
    )
else:
    print("Упорядоченных троек, удовлетворяющих порогам, не найдено.")

### 3.5 Распределение временных лагов между ФП

Медианные и средние временные лаги (дни) для наиболее частых упорядоченных пар. Показывает, как быстро один ФП следует за другим.

In [ ]:
# Временные лаги для значимых упорядоченных пар: дефолтные vs недефолтные
if len(pair_seq_stats) > 0:
    sig_pairs = pair_seq_stats[pair_seq_stats.get("Значимый", False)].head(10)
    if len(sig_pairs) == 0:
        sig_pairs = pair_seq_stats.head(10)

    lag_comparison = []
    for _, row in sig_pairs.iterrows():
        subset = ordered_pairs[
            (ordered_pairs["fp_type_1"] == row["fp_type_1"]) &
            (ordered_pairs["fp_type_2"] == row["fp_type_2"])
        ]
        for flag, label in [(1, "Дефолтные"), (0, "Недефолтные")]:
            lags = subset[subset["default_flag"] == flag]["lag_days"]
            if len(lags) > 0:
                lag_comparison.append({
                    "Последовательность": f"{fp_label(row['fp_type_1'])} → {fp_label(row['fp_type_2'])}",
                    "Группа": label,
                    "N": len(lags),
                    "Медиана (дни)": lags.median(),
                    "Среднее (дни)": lags.mean(),
                    "Мин (дни)": lags.min(),
                    "Макс (дни)": lags.max(),
                })

    df_lag = pd.DataFrame(lag_comparison)
    if len(df_lag) > 0:
        print("Временные лаги (A → B): дефолтные vs недефолтные")
        print("="*70)
        display(
            df_lag.style.hide(axis="index")
            .format({"Медиана (дни)": "{:.0f}", "Среднее (дни)": "{:.0f}",
                      "Мин (дни)": "{:.0f}", "Макс (дни)": "{:.0f}"})
        )

# Гистограмма лагов: дефолтные vs недефолтные (все пары)
if len(ordered_pairs) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    max_lag = int(ordered_pairs["lag_days"].quantile(0.95))
    bins = range(0, max_lag + 30, 30)

    def_lags = ordered_pairs[ordered_pairs["default_flag"] == 1]["lag_days"].clip(upper=max_lag)
    nodef_lags = ordered_pairs[ordered_pairs["default_flag"] == 0]["lag_days"].clip(upper=max_lag)

    ax.hist(nodef_lags, bins=bins, alpha=0.7, density=True, label="Недефолтные", color="steelblue", edgecolor="white")
    ax.hist(def_lags, bins=bins, alpha=0.7, density=True, label="Дефолтные", color="tomato", edgecolor="white")
    ax.set_xlabel("Лаг между ФП (дни)")
    ax.set_ylabel("Плотность")
    ax.set_title("Распределение временных лагов между последовательными ФП")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Анализ по сегментам (X_AREA_RESP)

Повторяем анализ комбинаций и последовательностей отдельно для каждого сегмента:
1. **ДКБ** — крупный бизнес
2. **ДМСБ (микро)** — микробизнес
3. **ДМСБ (малый + средний)** — малый и средний бизнес
4. **ДРПА**

### 4.1 Комбинации ФП по сегментам

In [ ]:
SEGMENTS = [
    ("ДКБ",                    dataset["X_AREA_RESP"] == "ДКБ"),
    ("ДМСБ (микро)",           dataset["X_AREA_RESP"] == "ДМСБ (микро)"),
    ("ДМСБ (малый + средний)", dataset["X_AREA_RESP"].isin(["ДМСБ (малый)", "ДМСБ (средний)"])),
    ("ДРПА",                   dataset["X_AREA_RESP"] == "ДРПА"),
]

seg_combo_results = {}

for seg_name, seg_mask in SEGMENTS:
    seg_data = dataset[seg_mask]
    n_seg = len(seg_data)
    n_def_seg = int(seg_data["default_flag"].sum())

    print(f"\n{'='*70}")
    if n_seg == 0:
        print(f"{seg_name}: 0 компаний — пропускаем")
        continue
    seg_rate = seg_data["default_flag"].mean()
    print(f"{seg_name}: {n_seg:,} компаний, {n_def_seg:,} дефолтных ({seg_rate:.2%} DR)")
    print("="*70)

    if n_seg < 30 or n_def_seg < 3:
        print("  Недостаточно данных для статистического анализа")
        continue

    # --- Индивидуальные ФП ---
    seg_fp_cols = [c for c in fp_cols if seg_data[c].sum() >= 3]
    df_ind_seg = compute_fp_stats(seg_data, seg_fp_cols, seg_rate, min_companies=3)

    n_sig_seg = df_ind_seg["Значимый (FDR<0.05)"].sum() if len(df_ind_seg) > 0 else 0
    print(f"\n  Индивидуальные ФП: {len(df_ind_seg)} проверено, {n_sig_seg} значимых")

    if len(df_ind_seg) > 0:
        df_ind_seg_show = df_ind_seg.head(10).copy()
        df_ind_seg_show.insert(1, "Наименование", df_ind_seg_show["ФП"].apply(
            lambda x: fp_name_map.get(str(x).strip(), "")
        ))
        print(f"\n  ТОП-10 ФП в сегменте {seg_name}:")
        display(
            df_ind_seg_show
            .style.hide(axis="index")
            .format({
                "Confidence": "{:.2%}", "Support": "{:.4f}",
                "p-value (Fisher)": "{:.2e}", "p-value (FDR)": "{:.2e}",
            })
        )

    # --- Пары ФП ---
    seg_frequent = [c for c in fp_cols if seg_data[c].sum() >= 10]
    if len(seg_frequent) >= 2:
        y_seg = seg_data["default_flag"].values
        X_seg = seg_data[seg_frequent].values
        names_seg = np.array(seg_frequent)

        seg_pairs = []
        for i, j in combinations(range(len(seg_frequent)), 2):
            mask = (X_seg[:, i] == 1) & (X_seg[:, j] == 1)
            n_w = mask.sum()
            if n_w < 5:
                continue
            nd = y_seg[mask].sum()
            if nd < 2:
                continue
            conf = nd / n_w
            seg_pairs.append({
                "Комбинация": f"{fp_label(names_seg[i])}  +  {fp_label(names_seg[j])}",
                "Компаний": n_w, "Дефолтов": nd,
                "Confidence": round(conf, 4),
                "Lift": round(conf / seg_rate, 2),
            })

        df_seg_pairs = pd.DataFrame(seg_pairs).sort_values("Lift", ascending=False)
        print(f"\n  Пар ФП: {len(df_seg_pairs)}")
        if len(df_seg_pairs) > 0:
            print(f"  ТОП-5 пар в сегменте {seg_name}:")
            display(
                df_seg_pairs.head(5)
                .style.hide(axis="index")
                .format({"Confidence": "{:.2%}"})
            )

    seg_combo_results[seg_name] = {
        "individual": df_ind_seg,
        "pairs": df_seg_pairs if "df_seg_pairs" in dir() else pd.DataFrame(),
    }

### 4.2 Последовательности ФП по сегментам

Повторяем анализ упорядоченных пар (A→B) для каждого сегмента.

In [ ]:
# Присоединяем сегмент к fp_first для фильтрации
fp_first_seg = fp_first.merge(
    dataset[["inn", "X_AREA_RESP"]], on="inn", how="left"
)

for seg_name, seg_mask in SEGMENTS:
    seg_inns = set(dataset[seg_mask]["inn"])
    fp_seg = fp_first_seg[fp_first_seg["inn"].isin(seg_inns)]

    n_seg = len(seg_inns)
    n_def_seg = int(dataset[seg_mask]["default_flag"].sum())

    print(f"\n{'='*70}")
    if n_seg < 30 or n_def_seg < 3:
        print(f"{seg_name}: недостаточно данных ({n_seg} компаний, {n_def_seg} дефолтов)")
        continue

    seg_rate = n_def_seg / n_seg
    print(f"{seg_name}: последовательности ФП ({n_seg:,} компаний, {seg_rate:.2%} DR)")
    print("="*70)

    # Self-join для упорядоченных пар
    op_seg = fp_seg.merge(fp_seg, on=["inn", "default_flag"], suffixes=("_1", "_2"))
    op_seg = op_seg[
        (op_seg["fp_type_1"] != op_seg["fp_type_2"]) &
        (op_seg["fp_date_1"] < op_seg["fp_date_2"])
    ].copy()
    op_seg["lag_days"] = (op_seg["fp_date_2"] - op_seg["fp_date_1"]).dt.days

    ps_seg = (
        op_seg.groupby(["fp_type_1", "fp_type_2"])
        .agg(
            n_companies=("inn", "nunique"),
            n_defaults=("default_flag", "sum"),
            median_lag=("lag_days", "median"),
        )
        .reset_index()
    )
    ps_seg = ps_seg[ps_seg["n_companies"] >= 5].copy()

    if len(ps_seg) == 0:
        print("  Упорядоченных пар с ≥5 компаниями не найдено")
        continue

    ps_seg["confidence"] = ps_seg["n_defaults"] / ps_seg["n_companies"]
    ps_seg["lift"] = ps_seg["confidence"] / seg_rate
    ps_seg = ps_seg.sort_values("lift", ascending=False)

    print(f"  Упорядоченных пар: {len(ps_seg)}")
    print(f"\n  ТОП-10 последовательностей (A → B) в сегменте {seg_name}:")
    show = ps_seg.head(10).copy()
    show["Последовательность"] = show["fp_type_1"].apply(fp_label) + " → " + show["fp_type_2"].apply(fp_label)
    display(
        show[["Последовательность", "n_companies", "n_defaults", "confidence", "lift", "median_lag"]]
        .rename(columns={
            "n_companies": "Компаний", "n_defaults": "Дефолтов",
            "confidence": "Confidence", "lift": "Lift", "median_lag": "Лаг (дни)",
        })
        .style.hide(axis="index")
        .format({"Confidence": "{:.2%}", "Лаг (дни)": "{:.0f}"})
    )

## 5. Сохранение результатов

In [ ]:
excel_path = f"{DATA_DIR}/hypothesis_1_results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_individual.to_excel(writer, sheet_name="Индивидуальные ФП", index=False)

    if len(df_pairs) > 0:
        df_pairs.to_excel(writer, sheet_name="Пары ФП", index=False)

    if len(df_triples) > 0:
        df_triples.to_excel(writer, sheet_name="Тройки ФП", index=False)

    if len(pair_seq_stats) > 0:
        ps_out = pair_seq_stats.copy()
        ps_out["Последовательность"] = ps_out["fp_type_1"] + " → " + ps_out["fp_type_2"]
        ps_out.to_excel(writer, sheet_name="Последовательности (пары)", index=False)

    if len(triple_seq_stats) > 0:
        ts_out = triple_seq_stats.copy()
        ts_out["Цепочка"] = ts_out["fp_type_1"] + " → " + ts_out["fp_type_2"] + " → " + ts_out["fp_type_3"]
        ts_out.to_excel(writer, sheet_name="Последовательности (тройки)", index=False)

    for seg_name, res in seg_combo_results.items():
        sheet_safe = seg_name[:28]
        if len(res.get("individual", [])) > 0:
            res["individual"].to_excel(writer, sheet_name=f"{sheet_safe}_ФП", index=False)

print(f"Результаты сохранены: {excel_path}")
print(f"\nЛисты:")
print(f"  • Индивидуальные ФП — {len(df_individual)} ФП с Lift, Confidence, p-value")
print(f"  • Пары ФП — {len(df_pairs)} комбинаций")
print(f"  • Тройки ФП — {len(df_triples)} комбинаций")
print(f"  • Последовательности (пары) — {len(pair_seq_stats)} упорядоченных пар")
print(f"  • + листы по сегментам")